In [1]:
import pandas as pd
import numpy as np
import sklearn
import xgboost

print("Everything is installed successfully!")

Everything is installed successfully!


In [2]:
import pandas as pd

old_data = pd.read_csv("Year_2009_2010_Cleaned.csv")
new_data = pd.read_csv("Year_2010_2011_Cleaned.csv")

print("Old Dataset Shape:", old_data.shape)
print("New Dataset Shape:", new_data.shape)

old_data.head()

Old Dataset Shape: (406876, 12)
New Dataset Shape: (541910, 13)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Customer_Segment,Sales_Channel,Payment_Mode,Order_Priority
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Regular,Online,UPI,High
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Regular,Online,UPI,Low
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Premium,Offline,Card,Medium
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,New,Retail,Cash,Low
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Regular,Online,Net Banking,High


In [3]:
print("Checking Data Drift...")

old_quantity_mean = old_data["Quantity"].mean()
new_quantity_mean = new_data["Quantity"].mean()

print("Old Quantity Mean:", old_quantity_mean)
print("New Quantity Mean:", new_quantity_mean)

if abs(old_quantity_mean - new_quantity_mean) > 1:
    drift_detected = True
    print("\n✅ Data Drift Detected")
else:
    drift_detected = False
    print("\n✅ No Data Drift Detected")

Checking Data Drift...
Old Quantity Mean: 13.609473156440783
New Quantity Mean: 9.552233765754462

✅ Data Drift Detected


In [4]:
new_data["Sales"] = new_data["Quantity"] * new_data["Price"]

X = new_data[["Quantity", "Price"]]
y = new_data["Sales"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data prepared successfully!")

Training data prepared successfully!


In [5]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

if drift_detected:

    print("Retraining Model...")

    model = XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    print("\nModel Retrained Successfully!")
    print("Mean Squared Error:", mse)
    print("R2 Score:", r2)

else:
    print("No retraining required.")

Retraining Model...

Model Retrained Successfully!
Mean Squared Error: 64566.80926397601
R2 Score: 0.03859409103481337


In [6]:
import joblib

if drift_detected:
    joblib.dump(model, "Sales_Prediction_Model_Day13.pkl")
    print("Model saved successfully as Sales_Prediction_Model_Day13.pkl")

Model saved successfully as Sales_Prediction_Model_Day13.pkl


In [7]:
print("\n========== AUTOMATED RETRAINING PIPELINE ==========")
print("1. Loaded old and new datasets")
print("2. Compared Quantity mean to detect drift")
print("3. Drift status:", drift_detected)

if drift_detected:
    print("4. Retrained XGBoost model")
    print("5. Saved updated model")
else:
    print("4. No retraining needed")

print("Pipeline Completed Successfully!")


========== AUTOMATED RETRAINING PIPELINE ==========
1. Loaded old and new datasets
2. Compared Quantity mean to detect drift
3. Drift status: True
4. Retrained XGBoost model
5. Saved updated model
Pipeline Completed Successfully!
